In [ ]:
import pandas as pd
import ast
import re
import random
import io
from google.colab import files

# ==========================================
# SECTION 1: DEFINITIONS
# ==========================================

def get_manual_tag_mapping(keywords_df):
    """Creates a searchable dictionary from the keywords CSV."""
    return dict(zip(keywords_df['keyword'].str.lower(), keywords_df['tag_name']))

def fill_tags_from_text(text, mapping):
    """Scans card text for keywords and returns corresponding tags."""
    if pd.isna(text): return []
    text_lower = str(text).lower()
    found_tags = []
    for kw, tag in mapping.items():
        # Clean the keyword for regex (handling simple wildcards like *)
        clean_kw = re.escape(kw).replace(r'\*', r'.*')
        if re.search(r'\b' + clean_kw + r'\b', text_lower):
            found_tags.append(tag)
    return list(set(found_tags))

def process_and_mark_empty(tag_list):
    """Ensures tags are lists and marks [] as EMPTY_DATA for the model."""
    if not tag_list or (isinstance(tag_list, list) and len(tag_list) == 0):
        return ["EMPTY_DATA"]
    return tag_list

def generate_test_tags():
    """Generates random high-dimensional tags for model testing."""
    pool = ["recursive-threat", "mana-fixing", "win-con-protection",
            "aggro-enabler", "midrange-value", "top-end-finisher",
            "engine-piece", "interaction-suite", "tempo-play"]
    return random.sample(pool, k=random.randint(1, 2))

# ==========================================
# FIXED SECTION 2: SMART FILENAME EXECUTION
# ==========================================

# 1. Upload both files
print("Upload your deck CSV and keywords CSV:")
uploaded = files.upload()

# Initialize variables
df = None
keywords = None

# 2. Smart Detection Loop
for filename in uploaded.keys():
    if "deck" in filename.lower():
        print(f"📂 Detected Deck File: {filename}")
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
    elif "keyword" in filename.lower():
        print(f"📂 Detected Keywords File: {filename}")
        keywords = pd.read_csv(io.BytesIO(uploaded[filename]))

# 3. Validation Check
if df is None or keywords is None:
    raise ValueError("Missing files! Ensure one filename contains 'deck' and the other 'keyword'.")

# 4. Initialize Mapping
tag_map = get_manual_tag_mapping(keywords)

# 5. Processing Loop
print("\nFilling blanks and generating testing data...")
final_rows = []

for idx, row in df.iterrows():
    # A. Handle Manual Tags
    # We check if it's already a list or a string representation of a list
    raw_man = row['manual_tags']
    man_tags = ast.literal_eval(raw_man) if isinstance(raw_man, str) and raw_man.startswith('[') else []

    if not man_tags or man_tags == []:
        man_tags = fill_tags_from_text(row['effects_line'], tag_map)

    # B. Fill Generated Tags (Randomly for testing)
    raw_gen = row['generated_effect_tags']
    gen_tags = ast.literal_eval(raw_gen) if isinstance(raw_gen, str) and raw_gen.startswith('[') else []

    if not gen_tags or gen_tags == []:
        gen_tags = generate_test_tags()

    # C. Apply the EMPTY_DATA marking
    row['manual_tags'] = process_and_mark_empty(man_tags)
    row['generated_effect_tags'] = process_and_mark_empty(gen_tags)

    final_rows.append(row)

# 6. Final Output
df_filled = pd.DataFrame(final_rows)
df_filled.to_csv('mtg_testing_data_filled.csv', index=False)

print("\n✅ Success! Created 'mtg_testing_data_filled.csv'")
print(f"Sample Card: {df_filled.iloc[0]['name']}")
print(f"Manual Tags: {df_filled.iloc[0]['manual_tags']}")